In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import gradio as gr

In [ ]:
class_map = {0: 'Cat', 1: 'Dog'}
model = tf.keras.models.load_model("../trained_model/bestmodel.keras")

def predict_detection(input_image: Image.Image):
    img_resized = input_image.resize((128, 128))
    image_array = tf.keras.utils.img_to_array(img_resized) / 255.0
    image_batch = np.expand_dims(image_array, axis=0)

    class_pred, box_pred = model.predict(image_batch)

    predicted_class_id = np.argmax(class_pred[0])
    confidence = float(class_pred[0][predicted_class_id])
    predicted_class_name = class_map[predicted_class_id]

    xmin, ymin, xmax, ymax = box_pred[0]

    cv_img = cv2.cvtColor(np.array(input_image), cv2.COLOR_RGB2BGR)
    h, w = cv_img.shape[:2]

    x1 = int(xmin * w)
    y1 = int(ymin * h)
    x2 = int(xmax * w)
    y2 = int(ymax * h)

    cv2.rectangle(cv_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    label = f"{predicted_class_name}: {confidence:.2f}"
    cv2.putText(cv_img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    output_image = Image.fromarray(cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB))
    return output_image


def build_interface():
    interface = gr.Interface(
        fn=predict_detection,
        inputs=gr.Image(type="pil"),
        outputs=gr.Image(type="pil"),
        title="Object Detection: Cats vs Dogs",
        description="Upload an image of a cat or dog, and see the detection box and classification."
    )
    return interface

In [ ]:
app = build_interface()
app.launch()